# **데이터 전처리**
* 목표 : 여행자와 여행지 seq 시계열 모델 학습용 분리

## 0. Import & Setting

In [51]:
import os 
import pandas as pd 
from pathlib import Path
from IPython.display import display, Markdown

In [3]:
ROOT_PATH = Path.cwd().parent
DATA_PATH = ROOT_PATH/"data"
CENTRAL_PATH = DATA_PATH/"raw/central/145.국내 여행로그 데이터_수도권_2차년도/3.개방데이터/1.데이터/Training/02.라벨링데이터/TL_CSV/"
EAST_PATH = DATA_PATH/"raw/east/146.국내 여행로그 데이터_동부권_2차년도/3.개방데이터/1.데이터/Training/02.라벨링데이터/TL_CSV/"
WEST_PATH = DATA_PATH/"raw/west/147.국내 여행로그 데이터_서부권_2차년도/3.개방데이터/1.데이터/Training/02.라벨링데이터/TL_CSV/"

assert all(path.exists() and len(list(path.iterdir()))==14 \
    for path in [CENTRAL_PATH,EAST_PATH,WEST_PATH]),\
    "경로 오류 또는 csv 파일 불일치"

In [ ]:
# 컬럼 이름 맞춰서 3개 df 통합시켜서 분석 가능한가? TRAVEL_ID 안겹치는지 확인
move_df_E = pd.read_csv(CENTRAL_PATH/"tn_move_his_이동내역_E.csv")
move_df_F = pd.read_csv(EAST_PATH/"tn_move_his_이동내역_F.csv")
move_df_G = pd.read_csv(WEST_PATH/"tn_move_his_이동내역_G.csv")

# 각 df의 TRAVEL_ID 집합 생성
travel_ids_E = set(move_df_E["TRAVEL_ID"].dropna().unique())
travel_ids_F = set(move_df_F["TRAVEL_ID"].dropna().unique())
travel_ids_G = set(move_df_G["TRAVEL_ID"].dropna().unique())

# 세 df 모두에 동시에 존재하는 TRAVEL_ID
common_all = travel_ids_E & travel_ids_F & travel_ids_G

# 두 개 이상 df에 중복으로 존재하는 TRAVEL_ID
duplicated_ids = (
    (travel_ids_E & travel_ids_F)
    | (travel_ids_E & travel_ids_G)
    | (travel_ids_F & travel_ids_G)
)

assert not duplicated_ids, f"TRAVEL_ID 독립적이지 않음 {set(duplicated_ids)}"

In [ ]:
# 컬럼 이름 맞춰서 3개 df 통합시켜서 분석 가능한가? TRAVEL_ID 안겹치는지 확인
move_df_E = pd.read_csv(CENTRAL_PATH/"tn_move_his_이동내역_E.csv")
move_df_F = pd.read_csv(EAST_PATH/"tn_move_his_이동내역_F.csv")
move_df_G = pd.read_csv(WEST_PATH/"tn_move_his_이동내역_G.csv")

# 각 df의 TRAVEL_ID 집합 생성
travel_ids_E = set(move_df_E["TRAVEL_ID"].dropna().unique())
travel_ids_F = set(move_df_F["TRAVEL_ID"].dropna().unique())
travel_ids_G = set(move_df_G["TRAVEL_ID"].dropna().unique())

# 세 df 모두에 동시에 존재하는 TRAVEL_ID
common_all = travel_ids_E & travel_ids_F & travel_ids_G

# 두 개 이상 df에 중복으로 존재하는 TRAVEL_ID
duplicated_ids = (
    (travel_ids_E & travel_ids_F)
    | (travel_ids_E & travel_ids_G)
    | (travel_ids_F & travel_ids_G)
)

assert not duplicated_ids, f"TRAVEL_ID 독립적이지 않음 {set(duplicated_ids)}"

In [10]:
'''
central : E 
east : F
west : G 
'''

for file in list(WEST_PATH.iterdir()):
    print(file.name)

tn_lodge_consume_his_숙박소비내역_G.csv
tn_activity_consume_his_활동소비내역_G.csv
tn_activity_his_활동내역_G.csv
tn_mvmn_consume_his_이동수단소비내역_G.csv
tc_sgg_시군구코드.csv
tc_codea_코드A.csv
tn_adv_consume_his_사전소비내역_G.csv
tn_travel_여행_G.csv
tn_visit_area_info_방문지정보_G.csv
tn_companion_info_동반자정보_G.csv
tn_move_his_이동내역_G.csv
tn_tour_photo_관광사진_G.csv
tn_traveller_master_여행객 Master_G.csv
tc_codeb_코드B.csv


## 1. EDA

In [57]:
# 공용 함수 

'''
make_name_efg 
df_name을 efg별 csv 파일 경로로 확장시키는 함수 
'''

def make_name_efg(df_name:str):
    e_path = CENTRAL_PATH/(df_name+"_E.csv")
    f_path = EAST_PATH/(df_name+"_F.csv")
    g_path = WEST_PATH/(df_name+"_G.csv")
    
    return e_path,f_path,g_path

'''
print_col_efg 
df_name을 가지는 df를 e,f,g 출력하는 함수

df_name : csv 확장자 제외한 파일 이름 
''' 
def print_col_efg(df_name:str):
    
    e_path,f_path,g_path = make_name_efg(df_name)
    
    e_df = pd.read_csv(e_path)
    f_df = pd.read_csv(f_path)
    g_df = pd.read_csv(g_path)
    
    if(e_df.columns.equals(f_df.columns) and f_df.columns.equals(g_df.columns)): 
        display(Markdown(f"### ===EFG 공통 : {df_name}==="))
        print(e_df.columns)
    else:
        display(Markdown(f"### ==={df_name}==="))
        common_cols = set(e_df.columns) & set(f_df.columns) & set(g_df.columns)
        all_cols = set(e_df.columns) | set(f_df.columns) | set(g_df.columns)
        print(f"교집합 컬럼 : {common_cols}")
        print(f"차집합 컬럼 : {all_cols-common_cols}")
        
'''
concat_efg 
컬럼이 모두 같은 3개의 EFG 파일 통합시키는 메서드 

df_name : csv 확장자 제외한 파일 이름 
'''
def concat_efg(df_name:str):
    e_path,f_path,g_path = make_name_efg(df_name)
        
    e_df = pd.read_csv(e_path)
    f_df = pd.read_csv(f_path)
    g_df = pd.read_csv(g_path)
    
    total_df = pd.concat([e_df,f_df,g_df],
                         axis = 0, 
                         ignore_index = True)
    
    return total_df 
    
    


### (0) 병합 필요 여부 검사
- CODE A, B는 여행 특성, 방문 목적 등을 코드로 분류한 표기표로 W,E,C 모두 동일 -> 병합할 필요 없음
- SGG CODE 또한 동일 -> 병합 필요 없음 
- 그 외 여행 데이터 csv는 모두 TRAVEL ID를 기준으로 구분됨 

#### (a) CODE A, B, SGG

In [62]:
## 코드 A 데이터를 병합할 필요가 있을지 확인 

c_codea_df = pd.read_csv(CENTRAL_PATH/"tc_codea_코드A.csv")
w_codea_df = pd.read_csv(WEST_PATH/"tc_codea_코드A.csv") 
e_codea_df = pd.read_csv(EAST_PATH/"tc_codea_코드A.csv")

## cd_nm이 모두 동일하다면 굳이 병합할 필요 없음
c_cd_nm = set(c_codea_df["cd_nm"].unique())
w_cd_nm = set(w_codea_df["cd_nm"].unique())
e_cd_nm = set(e_codea_df["cd_nm"].unique())

assert c_cd_nm == w_cd_nm and c_cd_nm == e_cd_nm,\
    "cd_nm 동일하지 않음"

print(c_cd_nm)

{'여행빈도_기간', '최종학력이수여부', '추천의향 점수', '여행종류', '여행동기', '직업기타', '소득', '메타데이터 진행상태', '방문선택이유', '여행동반자관계', '취식 지출 여부', '동반자동반상황', '여행스타일', '전반적 만족도 점수', '최종학력', '결제 방식', '주요 이동수단', '혼인상태', '성별', '방문활동', '방문지유형코드', '재방문 의향 점수', '여행상태', '동반자연령대', '미션', '직업', '여행권역', '재방문여부', '숙소유형', '입장료구분'}


In [63]:
c_codea_df.head()

,idx,cd_a,cd_nm,cd_memo,cd_memo2,del_flag,order_num,perm_write,perm_edit,perm_delete,ins_dt,edit_dt
0,5,STA,여행상태,NaN,NaN,N,999,N,N,N,2022-07-05 10:33:02,NaN
1,6,VIS,방문지유형코드,NaN,NaN,N,999,N,N,N,2022-07-05 10:45:53,NaN
2,7,MOV,주요 이동수단,NaN,NaN,N,999,N,N,N,2022-07-05 10:48:39,NaN
3,8,REV,재방문여부,NaN,NaN,N,999,N,N,N,2022-07-05 11:27:33,NaN
4,9,REN,방문선택이유,NaN,NaN,N,999,N,N,N,2022-07-05 11:28:18,NaN


In [67]:
c_codea_df[c_codea_df["cd_a"]=="MIS"]

,idx,cd_a,cd_nm,cd_memo,cd_memo2,del_flag,order_num,perm_write,perm_edit,perm_delete,ins_dt,edit_dt
14,19,MIS,미션,"여행테이블 여행목적, 여행미션에서 사용",NaN,N,999,N,N,N,2022-07-27 16:26:46,NaN


In [64]:
## 코드 B 데이터를 병합할 필요가 있을지 확인 

c_codeb_df = pd.read_csv(CENTRAL_PATH/"tc_codeb_코드B.csv")
w_codeb_df = pd.read_csv(WEST_PATH/"tc_codeb_코드B.csv") 
e_codeb_df = pd.read_csv(EAST_PATH/"tc_codeb_코드B.csv")

## cd_nm이 모두 동일하다면 굳이 병합할 필요 없음
c_cd_nm = set(c_codeb_df["cd_nm"].unique())
w_cd_nm = set(w_codeb_df["cd_nm"].unique())
e_cd_nm = set(e_codeb_df["cd_nm"].unique())

assert c_cd_nm == w_cd_nm and c_cd_nm == e_cd_nm,\
    "cd_nm 동일하지 않음"
print(c_cd_nm)

{'수료', '경남', '제주', '자연관광지', '장치․기계 조작 및 조립 종사자', '콘도미니엄/리조트', '월평균 100만원 ~ 200만원 미만', '렌터카(승용/승합/버스 등등)', '사별', '보통', '쇼핑', '일상적인 환경 및 역할에서의 탈출, 지루함 탈피', '캠핑카(자차 및 렌탈)', '현금', '시외/고속버스', '야외 스포츠, 레포츠 활동', '기능원 및 관련 기능 종사자', '도시 선호 매우선호', '전통 숙박시설', '캠핑', '취식', '1주일 ', '환승/경유', '지나가다 우연히', '휴식', '기타 활동', '매우 그렇다', '운동, 건강 증진 및 충전', '항공기', '이 방문지로 이동 중 구매/ 이 방문지로 배달', '새로운 경험 추구', '지역 축제/행사', '산책로, 둘레길 등', '월평균 600만원 ~ 700만원 미만', '반려동물 동반 여행', '택시', '관리자', '여자', '경북', '도서지역', '사진 업로드 전', '여행 출발 전 거주지 인근에서 미리 구매', '상점', '과거 경험이 좋아서', '강원', '호캉스 여행', '승인 거절', '자연 선호 중간선호', '농림어업 숙련 종사자', '친인척', '여행 후(기록전)', '매우 불만족', '교육/체험 프로그램 참가', 'Well-ness 여행', '교육성이 좋아서', '인플루언서 따라하기 여행', '버스 + 지하철', '예정여행지변경신청', '충북', '친구', '가성비가 좋아서', '연인', '1년', '최초 로그인', '배우자', '~9세이하', '월평균 900만원 ~ 1,000만원 미만', '월평균 700만원 ~ 800만원 미만', '지명도/명소/핫플레이스', '불만족', '20대', '전혀 그렇지 않다', '여행 중', '시티투어', '친목 단체/모임(동호회, 종교단체 등)', '승인 진행 중', '여행 기록 보완 완료', '대학교(4년제 이상)', '여행 중 이탈', '기타', '단순노무종사자', '가기 편해서/교통이 좋아서', '10대', '무

In [65]:
c_codeb_df.head()

,idx,cd_a,cd_b,cd_nm,cd_memo,cd_memo2,del_flag,order_num,ins_dt,edit_dt
0,1055,ACT,1,취식,NaN,NaN,N,10,2022-07-05 11:34:29,NaN
1,1056,ACT,2,쇼핑 / 구매,(아이 쇼핑 포함)<br>* 이전 방문지에서 해당 방문지 이동 중 구매한 내역 포함,NaN,N,20,2022-07-05 11:49:08,NaN
2,1057,ACT,3,체험 활동 / 입장 및 관람,NaN,NaN,N,30,2022-07-05 11:49:37,NaN
3,1058,ACT,4,단순 구경 / 산책 / 걷기,NaN,NaN,N,40,2022-07-05 11:49:43,NaN
4,1059,ACT,5,휴식,NaN,NaN,N,50,2022-07-05 11:49:48,NaN


In [73]:
c_codeb_df[c_codeb_df["cd_a"]=="MIS"]

,idx,cd_a,cd_b,cd_nm,cd_memo,cd_memo2,del_flag,order_num,ins_dt,edit_dt
95,1097,MIS,1,쇼핑,NaN,NaN,N,999,2022-07-27 16:26:52,NaN
96,1098,MIS,2,"테마파크, 놀이시설, 동/식물원 방문",NaN,NaN,N,999,2022-07-27 16:26:58,NaN
97,1099,MIS,3,역사 유적지 방문,NaN,NaN,N,999,2022-07-27 16:27:03,NaN
98,1100,MIS,4,시티투어,NaN,NaN,N,999,2022-07-27 16:27:08,NaN
99,1101,MIS,5,"야외 스포츠, 레포츠 활동",NaN,NaN,N,999,2022-07-27 16:27:14,NaN
100,1102,MIS,6,지역 문화예술/공연/전시시설 관람,NaN,NaN,N,999,2022-07-27 16:27:19,NaN
101,1103,MIS,7,유흥/오락(나이트라이프),NaN,NaN,N,999,2022-07-27 16:27:26,NaN
102,1104,MIS,8,캠핑,NaN,NaN,N,999,2022-07-27 16:27:31,NaN
103,1105,MIS,9,지역 축제/이벤트 참가,NaN,NaN,N,999,2022-07-27 16:27:37,NaN
104,1106,MIS,10,온천/스파,NaN,NaN,N,999,2022-07-27 16:27:43,NaN


In [62]:
## 코드 B 데이터를 병합할 필요가 있을지 확인 

c_sgg_df = pd.read_csv(CENTRAL_PATH/"tc_sgg_시군구코드.csv")
w_sgg_df = pd.read_csv(WEST_PATH/"tc_sgg_시군구코드.csv") 
e_sgg_df = pd.read_csv(EAST_PATH/"tc_sgg_시군구코드.csv")

## cd_nm이 모두 동일하다면 굳이 병합할 필요 없음
c_sgg = set(c_sgg_df["SGG_CD"].unique())
w_sgg = set(w_sgg_df["SGG_CD"].unique())
e_sgg = set(e_sgg_df["SGG_CD"].unique())

assert c_sgg == w_sgg and c_sgg == e_sgg,\
    "시군구 코드 동일하지 않음"
print(c_sgg)

{np.int64(4617011200), np.int64(2771025921), np.int64(2771025922), np.int64(2771025923), np.int64(1114013700), np.int64(2771025924), np.int64(2771025925), np.int64(2771025926), np.int64(2771025927), np.int64(2771025928), np.int64(2771025929), np.int64(2771025930), np.int64(2771025931), np.int64(2771025932), np.int64(2771025933), np.int64(4215013400), np.int64(4219011100), np.int64(4812111900), np.int64(4159012900), np.int64(4163010600), np.int64(4611014700), np.int64(4615012400), np.int64(1120010300), np.int64(3023011900), np.int64(4128112700), np.int64(5011013700), np.int64(2826010700), np.int64(4217012300), np.int64(4729012300), np.int64(2911010900), np.int64(4678025300), np.int64(4613013600), np.int64(4617011300), np.int64(1114013800), np.int64(3171025000), np.int64(4678025321), np.int64(4678025322), np.int64(4575035500), np.int64(4678025323), np.int64(4678025324), np.int64(4678025325), np.int64(4678025326), np.int64(4678025327), np.int64(4678025329), np.int64(4678025330), np.int64(

#### (b) 여행 정보 
- tn_move_his_이동내역_E.csv
- tn_travel_여행_E.csv
- tn_activity_his_활동내역_E.csv
- tn_visit_area_info_방문지정보_E.csv

In [58]:
# 3개 csv의 컬럼 종류 파악하기 
# 이동내역 컬럼 
print_col_efg("tn_move_his_이동내역")
# 여행 컬럼 
print_col_efg("tn_travel_여행")
# 활동내역 컬럼 
print_col_efg("tn_activity_his_활동내역")
# 방문지 정보 
print_col_efg("tn_visit_area_info_방문지정보")

### ===EFG 공통 : tn_move_his_이동내역===

Index(['TRAVEL_ID', 'TRIP_ID', 'START_VISIT_AREA_ID', 'END_VISIT_AREA_ID',
       'START_DT_MIN', 'END_DT_MIN', 'MVMN_CD_1', 'MVMN_CD_2'],
      dtype='str')


### ===EFG 공통 : tn_travel_여행===

Index(['TRAVEL_ID', 'TRAVEL_NM', 'TRAVELER_ID', 'TRAVEL_PURPOSE',
       'TRAVEL_START_YMD', 'TRAVEL_END_YMD', 'MVMN_NM', 'TRAVEL_PERSONA',
       'TRAVEL_MISSION', 'TRAVEL_MISSION_CHECK'],
      dtype='str')


### ===EFG 공통 : tn_activity_his_활동내역===

Index(['TRAVEL_ID', 'VISIT_AREA_ID', 'ACTIVITY_TYPE_CD', 'ACTIVITY_TYPE_SEQ',
       'ACTIVITY_ETC', 'ACTIVITY_DTL', 'RSVT_YN', 'EXPND_SE', 'ADMISSION_SE'],
      dtype='str')


### ===EFG 공통 : tn_visit_area_info_방문지정보===

Index(['VISIT_AREA_ID', 'TRAVEL_ID', 'VISIT_ORDER', 'VISIT_AREA_NM',
       'VISIT_START_YMD', 'VISIT_END_YMD', 'ROAD_NM_ADDR', 'LOTNO_ADDR',
       'X_COORD', 'Y_COORD', 'ROAD_NM_CD', 'LOTNO_CD', 'POI_ID', 'POI_NM',
       'RESIDENCE_TIME_MIN', 'VISIT_AREA_TYPE_CD', 'REVISIT_YN',
       'VISIT_CHC_REASON_CD', 'LODGING_TYPE_CD', 'DGSTFN', 'REVISIT_INTENTION',
       'RCMDTN_INTENTION', 'SGG_CD'],
      dtype='str')


In [38]:
# 컬럼 이름 맞춰서 3개 df 통합시켜서 분석 가능한가? TRAVEL_ID 안겹치는지 확인
move_df_E = pd.read_csv(CENTRAL_PATH/"tn_move_his_이동내역_E.csv")
move_df_F = pd.read_csv(EAST_PATH/"tn_move_his_이동내역_F.csv")
move_df_G = pd.read_csv(WEST_PATH/"tn_move_his_이동내역_G.csv")

# 각 df의 TRAVEL_ID 집합 생성
travel_ids_E = set(move_df_E["TRAVEL_ID"].dropna().unique())
travel_ids_F = set(move_df_F["TRAVEL_ID"].dropna().unique())
travel_ids_G = set(move_df_G["TRAVEL_ID"].dropna().unique())

# 세 df 모두에 동시에 존재하는 TRAVEL_ID
common_all = travel_ids_E & travel_ids_F & travel_ids_G

# 두 개 이상 df에 중복으로 존재하는 TRAVEL_ID
duplicated_ids = (
    (travel_ids_E & travel_ids_F)
    | (travel_ids_E & travel_ids_G)
    | (travel_ids_F & travel_ids_G)
)

assert not duplicated_ids, f"TRAVEL_ID 독립적이지 않음 {set(duplicated_ids)}"

In [60]:
# E,F,G 통합시키기 
move_total_df = concat_efg("tn_move_his_이동내역")
display(Markdown("### 이동내역"))
display(move_total_df.head())

travel_total_df = concat_efg("tn_travel_여행")
display(Markdown("### 여행"))
display(travel_total_df.head())

activity_total_df = concat_efg("tn_activity_his_활동내역")
display(Markdown("### 활동내역"))
display(activity_total_df.head())

visit_area_total_df = concat_efg("tn_visit_area_info_방문지정보")
display(Markdown("### 방문지정보"))
display(visit_area_total_df.head())

### 이동내역

,TRAVEL_ID,TRIP_ID,START_VISIT_AREA_ID,END_VISIT_AREA_ID,START_DT_MIN,END_DT_MIN,MVMN_CD_1,MVMN_CD_2
0,e_e000004,2304300001,2.304300e+09,NaN,2023-04-30 13:30,NaN,NaN,NaN
1,e_e000004,2304300002,NaN,2.304300e+09,NaN,2023-04-30 14:00,1.0,NaN
2,e_e000004,2304300003,NaN,2.304300e+09,NaN,2023-04-30 15:00,15.0,NaN
3,e_e000004,2304300004,NaN,2.304300e+09,NaN,2023-04-30 15:30,15.0,NaN
4,e_e000004,2304300005,NaN,2.304300e+09,NaN,2023-04-30 17:30,1.0,NaN


### 여행

,TRAVEL_ID,TRAVEL_NM,TRAVELER_ID,TRAVEL_PURPOSE,TRAVEL_START_YMD,TRAVEL_END_YMD,MVMN_NM,TRAVEL_PERSONA,TRAVEL_MISSION,TRAVEL_MISSION_CHECK
0,e_e000004,E03,e000004,3,2023-04-30,2023-05-01,NaN,서울 외 수도권 방문/수도권 거주/40세 이상/자녀동반/일반미션,3,3;4;11
1,e_e000006,E03,e000006,21,2023-04-30,2023-05-02,NaN,경기 방문/거주지 구분 없음/39세 이하/특별미션,21,21;10;27
2,e_e000009,E03,e000009,2;4,2023-04-29,2023-05-01,NaN,서울 외 수도권 방문/수도권 거주/39세 이하/커플/일반미션,2;4,22;1;7
3,e_e000010,E01,e000010,3;6,2023-04-29,2023-05-01,NaN,서울 방문/수도권 외 거주/39세 이하/나홀로 여행/일반미션,3;6,6;3;1
4,e_e000011,E01,e000011,1;21,2023-04-28,2023-05-01,NaN,서울 방문/수도권 외 거주/40세 이상/커플/일반미션,1;21,6;2;24


### 활동내역

,TRAVEL_ID,VISIT_AREA_ID,ACTIVITY_TYPE_CD,ACTIVITY_TYPE_SEQ,ACTIVITY_ETC,ACTIVITY_DTL,RSVT_YN,EXPND_SE,ADMISSION_SE
0,e_e000004,2304300002,3,0,NaN,화성 어차 탑승,Y,5,1.0
1,e_e000004,2304300003,4,0,NaN,NaN,NaN,NaN,NaN
2,e_e000004,2304300004,4,0,NaN,NaN,NaN,NaN,NaN
3,e_e000006,2304300005,1,0,NaN,소금빵,N,1,NaN
4,e_e000006,2304300007,1,0,NaN,쭈꾸미,N,1,NaN


### 방문지정보

,VISIT_AREA_ID,TRAVEL_ID,VISIT_ORDER,VISIT_AREA_NM,VISIT_START_YMD,VISIT_END_YMD,ROAD_NM_ADDR,LOTNO_ADDR,X_COORD,Y_COORD,...,POI_NM,RESIDENCE_TIME_MIN,VISIT_AREA_TYPE_CD,REVISIT_YN,VISIT_CHC_REASON_CD,LODGING_TYPE_CD,DGSTFN,REVISIT_INTENTION,RCMDTN_INTENTION,SGG_CD
0,2304300001,e_e000004,1,집,2023-04-30,2023-04-30,NaN,NaN,NaN,NaN,...,NaN,NaN,21,NaN,NaN,NaN,NaN,NaN,NaN,4.159012e+09
1,2304300002,e_e000004,2,화성 관광열차 안내소 연무대 매표소,2023-04-30,2023-04-30,경기 수원시 팔달구 창룡대로103번길 20,경기 수원시 팔달구 매향동 3-32,127.023339,37.287878,...,동대문종합시장 악세서리부자재시장,60.0,2,N,10.0,NaN,4.0,3.0,4.0,NaN
2,2304300003,e_e000004,3,창룡문,2023-04-30,2023-04-30,NaN,경기 수원시 팔달구 남수동,127.025143,37.287791,...,창룡문,30.0,2,N,1.0,NaN,4.0,4.0,4.0,NaN
3,2304300004,e_e000004,4,수원 화성 화홍문,2023-04-30,2023-04-30,NaN,경기 수원시 팔달구 북수동 9000-1,127.017626,37.287546,...,수원화성 화홍문,60.0,2,N,10.0,NaN,4.0,3.0,3.0,NaN
4,2304300005,e_e000004,5,집,2023-04-30,2023-05-01,NaN,NaN,NaN,NaN,...,NaN,390.0,21,NaN,NaN,NaN,NaN,NaN,NaN,4.159012e+09


- 여행 특성 나타내는 카테고리 종류 :  TRAVEL_PURPOSE, TRAVEL_MISSION, TRAVEL_MISSION_CHECK
- TRAVEL_PURPOSE와 TRAVEL_MISSION은 동일함, TRAVEL_MISSION_CHECK는 여러개의 카테고리를 가짐
- 정확한 차이는 찾지 못했으나 아마 TRAVEL_MISSION_CHECK가 실제로 여행한 특성을 더 많이 나타내는 듯  
- TRAVEL_MISSION_CHECK가 가장 데이터가 풍부하므로 이를 선택하는 게 좋을 듯 

In [80]:
mission_map = {item["cd_b"]:item["cd_nm"] for _,item in c_codeb_df[c_codeb_df["cd_a"]=="MIS"].iterrows()}
print(mission_map)


{'1': '쇼핑', '2': '테마파크, 놀이시설, 동/식물원 방문', '3': '역사 유적지 방문', '4': '시티투어', '5': '야외 스포츠, 레포츠 활동', '6': '지역 문화예술/공연/전시시설 관람', '7': '유흥/오락(나이트라이프)', '8': '캠핑', '9': '지역 축제/이벤트 참가', '10': '온천/스파', '11': '교육/체험 프로그램 참가', '12': '드라마 촬영지 방문', '13': '종교/성지 순례', '21': 'Well-ness 여행', '22': 'SNS 인생샷 여행', '23': '호캉스 여행', '24': '신규 여행지 발굴', '25': '반려동물 동반 여행', '26': '인플루언서 따라하기 여행', '27': '친환경 여행(플로깅 여행)', '28': '등반 여행'}


- MIS 코드 분류 
~~~
{
    '1': '쇼핑', 
    '2': '테마파크, 놀이시설, 동/식물원 방문', 
    '3': '역사 유적지 방문', 
    '4': '시티투어', 
    '5': '야외 스포츠, 레포츠 활동', 
    '6': '지역 문화예술/공연/전시시설 관람', 
    '7': '유흥/오락(나이트라이프)', 
    '8': '캠핑', 
    '9': '지역 축제/이벤트 참가', 
    '10': '온천/스파', 
    '11': '교육/체험 프로그램 참가', 
    '12': '드라마 촬영지 방문', 
    '13': '종교/성지 순례', 
    '21': 'Well-ness 여행', 
    '22': 'SNS 인생샷 여행', 
    '23': '호캉스 여행', 
    '24': '신규 여행지 발굴', 
    '25': '반려동물 동반 여행', 
    '26': '인플루언서 따라하기 여행', 
    '27': '친환경 여행(플로깅 여행)', 
    '28': '등반 여행'}

~~~
- 야단법석 서비스 코드 분류 

~~~
mission_to_motive = {
    # 1. 지루한 일상 탈출
    "4": "지루한 일상 탈출",      # 시티투어
    "7": "지루한 일상 탈출",      # 유흥/오락

    # 2. 피로를 해소하는 휴식
    "10": "피로를 해소하는 휴식",  # 온천/스파
    "21": "피로를 해소하는 휴식",  # Well-ness 여행
    "23": "피로를 해소하는 휴식",  # 호캉스 여행

    # 3. 소중한 사람과의 추억
    "25": "소중한 사람과의 추억",  # 반려동물 동반 여행

    # 4. 나를 돌아보는 시간
    "13": "나를 돌아보는 시간",    # 종교/성지 순례
    "28": "나를 돌아보는 시간",    # 등반 여행

    # 5. 남는 건 사진 뿐
    "12": "남는 건 사진 뿐",       # 드라마 촬영지 방문
    "22": "남는 건 사진 뿐",       # SNS 인생샷 여행
    "26": "남는 건 사진 뿐",       # 인플루언서 따라하기 여행

    # 6. 액티비티로 활력 충전
    "2": "액티비티로 활력 충전",   # 테마파크, 놀이시설, 동/식물원 방문
    "5": "액티비티로 활력 충전",   # 야외 스포츠, 레포츠 활동
    "8": "액티비티로 활력 충전",   # 캠핑

    # 7. 낯선 곳에서의 설렘
    "9": "낯선 곳에서의 설렘",     # 지역 축제/이벤트 참가
    "11": "낯선 곳에서의 설렘",    # 교육/체험 프로그램 참가
    "24": "낯선 곳에서의 설렘",    # 신규 여행지 발굴
    "27": "낯선 곳에서의 설렘",    # 친환경 여행

    # 8. 우리 역사 문화 탐방
    "3": "우리 역사 문화 탐방",    # 역사 유적지 방문
    "6": "우리 역사 문화 탐방",    # 지역 문화예술/공연/전시시설 관람

    # 9. 특별한 기념일
    # 직접 대응되는 미션 코드는 약함. TRAVEL_MOTIVE 쪽 9번에 가까움.

    # 10. 발길이 이끄는 대로
    "1": "발길이 이끄는 대로",     # 쇼핑
}

~~~

In [ ]:
# 여행 계획 산출에 필요한 컬럼 
# TRAVEL_ID, VISIT_AREA_ID,VISIT_AREA_NM, VISIT_ORDER, CONTENT_ID, TRAVEL_PURPOSE, MTM (mission -> motive)

tn_mvmn_consume_his_이동수단소비내역_E.csv
<class 'pandas.DataFrame'>
RangeIndex: 4122 entries, 0 to 4121
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   TRAVEL_ID        4122 non-null   str    
 1   MVMN_SE          4122 non-null   int64  
 2   PAYMENT_SE       4122 non-null   str    
 3   PAYMENT_SEQ      4122 non-null   int64  
 4   MVMN_SE_NM       4122 non-null   str    
 5   RSVT_YN          4122 non-null   str    
 6   PAYMENT_NUM      4122 non-null   int64  
 7   BRNO             370 non-null    float64
 8   STORE_NM         378 non-null    str    
 9   PAYMENT_DT       416 non-null    str    
 10  PAYMENT_MTHD_SE  417 non-null    float64
 11  PAYMENT_AMT_WON  4122 non-null   int64  
 12  PAYMENT_ETC      406 non-null    str    
dtypes: float64(2), int64(4), str(7)
memory usage: 418.8 KB
None
tn_activity_his_활동내역_E.csv
<class 'pandas.DataFrame'>
RangeIndex: 23689 entries, 0 to 23688
Data

#### (c) 여행자 특성
- tn_traveller_master_여행객 Master_G.csv

In [82]:
# E,F,G 통합시키기 
traveler_df = concat_efg("tn_traveller_master_여행객 Master")
display(Markdown("### 여행자 특성"))
display(traveler_df.head())

### 여행자 특성

,TRAVELER_ID,RESIDENCE_SGG_CD,GENDER,AGE_GRP,EDU_NM,EDU_FNSH_SE,MARR_STTS,FAMILY_MEMB,JOB_NM,JOB_ETC,...,TRAVEL_STYL_7,TRAVEL_STYL_8,TRAVEL_STATUS_RESIDENCE,TRAVEL_STATUS_DESTINATION,TRAVEL_STATUS_ACCOMPANY,TRAVEL_STATUS_YMD,TRAVEL_MOTIVE_1,TRAVEL_MOTIVE_2,TRAVEL_MOTIVE_3,TRAVEL_COMPANIONS_NUM
0,e004720,41,여,60,4,1.0,3,3,11,NaN,...,5,5,경기도,서울,2인 가족 여행,2023-07-16~2023-07-16,2,6.0,NaN,1
1,e000914,30,여,20,6,1.0,1,1,3,NaN,...,4,1,대전광역시,서울,나홀로 여행,2023-06-03~2023-06-03,1,7.0,10.0,0
2,e003564,41,여,30,7,1.0,2,4,2,NaN,...,1,7,경기도,경기,자녀 동반 여행,2023-06-24~2023-06-24,8,3.0,7.0,3
3,e000396,41,여,30,6,1.0,2,2,2,NaN,...,1,6,경기도,인천,2인 가족 여행,2023-05-20~2023-05-21,9,1.0,7.0,1
4,e001890,11,남,20,6,1.0,1,4,3,NaN,...,5,6,서울특별시,경기,2인 여행(가족 외),2023-06-04~2023-06-04,3,1.0,5.0,1


In [83]:
traveler_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7680 entries, 0 to 7679
Data columns (total 36 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   TRAVELER_ID                7680 non-null   str    
 1   RESIDENCE_SGG_CD           7680 non-null   int64  
 2   GENDER                     7680 non-null   str    
 3   AGE_GRP                    7680 non-null   int64  
 4   EDU_NM                     7680 non-null   int64  
 5   EDU_FNSH_SE                7678 non-null   float64
 6   MARR_STTS                  7680 non-null   int64  
 7   FAMILY_MEMB                7680 non-null   int64  
 8   JOB_NM                     7680 non-null   int64  
 9   JOB_ETC                    0 non-null      float64
 10  INCOME                     7680 non-null   int64  
 11  HOUSE_INCOME               5929 non-null   float64
 12  TRAVEL_TERM                7680 non-null   int64  
 13  TRAVEL_NUM                 7680 non-null   int64  
 14  TRA

In [84]:
# RESIDENCE_SGG_CD	GENDER	AGE_GRP	EDU_NM
# TRAVEL_STYL_7	TRAVEL_STYL_8왜 나뉘는..????? 
# TRAVEL_STATUS_DESTINATION를 테마 여행 지표로 간주해도 될까? 
# TRAVEL_COMPANIONS_NUM
# TRAVEL_MOTIVE_1, TRAVEL_MOTIVE_2, TRAVEL_MOTIVE_3 가 뭐지 
# 회원가입 온보딩 : 거주 지역 / 여행 스타일 /선호 여행 지역 / 성별 
# 여행 온보딩 : barrier 조건 (아이 동반/어르신 동반/휠체어 이용) / 여행의 동기(10개) / 동반자 최대 2 

## 2. Preprocess

### (1) 지역 테마 여행 전처리
* 목표 : 서울, 인천, 대전, 수원, 광주, 부산, 대구, 창원 지역별 여행 별도의 df로 분리

### (2) 방문지 정보 전처리
* 목표 : visit_area_id와 contentid 매칭시키기

### (3) 방문자 정보 전처리
* 목표 : 동행자, 방문자 특징 추출해 지역별 csv 파일 정리

## 3. Visualization